<a href="https://colab.research.google.com/github/HArmor28/ML/blob/main/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HArmor28/ML/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

The source table contains one row per pseudonymized client, content item, and reporting date. For this contract, March 1–31, 2026 is the observed feature window, and April 1–30, 2026 is the future outcome window. After aggregation, one modeling row will represent one client-content item at the March 31 cutoff.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 1: verify the unit of analysis and time window

from google.colab import userdata
from IPython.display import display
import duckdb

# Read the token from Colab Secrets; never paste the token into this notebook.
hf_token = userdata.get("HF_TOKEN")
assert hf_token, "HF_TOKEN is unavailable. Enable notebook access in Colab Secrets."

# Connect DuckDB to the gated Hugging Face dataset.
con = duckdb.connect()
con.execute("INSTALL httpfs")
con.execute("LOAD httpfs")

# Escape the token for the SQL statement without displaying it.
safe_token = hf_token.replace("'", "''")
con.execute(f"""
    CREATE OR REPLACE SECRET hf_secret (
        TYPE huggingface,
        TOKEN '{safe_token}'
    )
""")

# Use a mid-panel month. June 2026 remains untouched for later testing.
march_path = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/month=2026-03/*.parquet"
)

# 1. Check row count and observed date window.
window_check = con.sql(f"""
    SELECT
        COUNT(*) AS row_count,
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date,
        COUNT(DISTINCT client_hash_id) AS client_count,
        COUNT(DISTINCT content_hash_id) AS content_count
    FROM read_parquet('{march_path}')
""").df()

print("Observed March 2026 window:")
display(window_check)

# 2. Check the documented grain:
# one report date × pseudonymized client × content item.
grain_violations = con.sql(f"""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        COUNT(*) AS copies
    FROM read_parquet('{march_path}')
    GROUP BY
        report_date,
        client_hash_id,
        content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

print("Duplicate grain combinations (expected: 0 rows):")
display(grain_violations)

# 3. Show the number of observed days per client.
# Different values demonstrate that the panel may be unbalanced.
client_coverage = con.sql(f"""
    SELECT
        client_hash_id,
        COUNT(DISTINCT report_date) AS observed_days,
        MIN(report_date) AS first_observed_date,
        MAX(report_date) AS last_observed_date
    FROM read_parquet('{march_path}')
    GROUP BY client_hash_id
    ORDER BY observed_days
    LIMIT 10
""").df()

print("Ten clients with the fewest observed days in this partition:")
display(client_coverage)

# Automated checks for the contract claims.
assert str(window_check.loc[0, "first_date"].date()) == "2026-03-01"
assert str(window_check.loc[0, "last_date"].date()) == "2026-03-31"
assert grain_violations.empty, "The documented source grain did not hold."

print("Checks passed: March window confirmed and no grain duplicates found.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Observed March 2026 window:


,row_count,first_date,last_date,client_count,content_count
0,9841378,2026-03-01,2026-03-31,55,331437


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate grain combinations (expected: 0 rows):


,report_date,client_hash_id,content_hash_id,copies


Ten clients with the fewest observed days in this partition:


,client_hash_id,observed_days,first_observed_date,last_observed_date
0,client_e00b29e582949543,9,2026-03-23,2026-03-31
1,client_810019792c9b8efc,12,2026-03-20,2026-03-31
2,client_f6f0cdf26d03d7bd,13,2026-03-19,2026-03-31
3,client_86ebc2f12c01f586,29,2026-03-03,2026-03-31
4,client_3ffa76342f366962,31,2026-03-01,2026-03-31
5,client_19b89ee4fe3db6da,31,2026-03-01,2026-03-31
6,client_08a6a72ff48e62c0,31,2026-03-01,2026-03-31
7,client_f623b01661d4bfe4,31,2026-03-01,2026-03-31
8,client_fef1a8f436438636,31,2026-03-01,2026-03-31
9,client_ba65e80a1116ae41,31,2026-03-01,2026-03-31


Checks passed: March window confirmed and no grain duplicates found.


## 2. Fields: feature / label / context / excluded

**Features:** I will use GSC, GA4, traffic, engagement, scroll, and data-availability measurements observed during the March feature window. These values must be available before the prediction date.

**Label / proxy:** `is_click_decline_next30` will indicate whether measured GSC clicks decline during the following 30-day outcome window. It is a defined proxy for directional decline, not proof that the content needs to be refreshed.

**Context:** `client_hash_id`, `content_hash_id`, and `report_date` will be used for grouping, joining, splitting, and defining time windows. They will not be model features.

**Excluded:** Future metrics are excluded because they would leak the outcome. Account-access fields are excluded because they do not measure content performance. `gsc_sum_position` is excluded as a separate feature because it is used to calculate average position. Client names, URLs, and raw queries are excluded for privacy.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
feature_sources = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_sessions",
    "ga4_engaged_sessions",
    "ga4_total_engagement_sec",
    "sessions_organic",
    "sessions_ai",
    "scroll_events",
    "gsc_data_available",
    "ga4_data_available",
]

label = ["is_click_decline_next30"]

context = [
    "client_hash_id",
    "content_hash_id",
    "report_date",
]

excluded = {
    "gsc_sum_position": "Redundant with average-position calculation",
    "client_has_gsc": "Account access, not content performance",
    "client_has_ga4": "Account access, not content performance",
    "future_metrics": "Unavailable at prediction time; leakage",
    "client_names_urls_queries": "Private or identifying",
}

# Verify that the required raw source columns exist.
schema = con.sql(f"""
    DESCRIBE
    SELECT *
    FROM read_parquet('{march_path}')
""").df()

available = set(schema["column_name"])
required = set(feature_sources + context)

missing = sorted(required - available)
assert not missing, f"Missing source fields: {missing}"

print("FEATURE SOURCES:", feature_sources)
print("LABEL / PROXY:", label)
print("CONTEXT:", context)
print("EXCLUDED:")
for field, reason in excluded.items():
    print(f"  {field}: {reason}")

print("\nAll required source fields were found.")

FEATURE SOURCES: ['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_ai', 'scroll_events', 'gsc_data_available', 'ga4_data_available']
LABEL / PROXY: ['is_click_decline_next30']
CONTEXT: ['client_hash_id', 'content_hash_id', 'report_date']
EXCLUDED:
  gsc_sum_position: Redundant with average-position calculation
  client_has_gsc: Account access, not content performance
  client_has_ga4: Account access, not content performance
  future_metrics: Unavailable at prediction time; leakage
  client_names_urls_queries: Private or identifying

All required source fields were found.


## 3. Verify it with queries (grain, counts, missing values, windows)

The March partition contains approximately 9.84 million observed daily content-performance rows across 55 pseudonymized clients and 331,437 content items. No duplicate combinations of report date, client ID, and content ID were observed.

GSC data was unavailable for 63.31% of March rows. Because `gsc_clicks` was never null, a non-null or zero click value does not prove that GSC activity was measured. GA4 data was unavailable for 65.12% of rows, while `ga4_sessions` was null for only 30.67%. Missing GA4 data is therefore not represented consistently by nulls. I will use `ga4_data_available` rather than treating every zero as measured zero activity.

Client histories are unbalanced. The median client has 31 observed March days, but the least-covered client has only nine. A March date filter therefore does not guarantee a complete 31-day history for every client. The proposed April outcome window exists and covers April 1–30, 2026.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 3: verify grain, counts, missingness, and windows

# Reuse the grain and count checks from Section 1.
print("March counts and window:")
display(window_check)

print("Grain violations (expected: 0 rows):")
display(grain_violations)

assert grain_violations.empty, "Duplicate grain rows were found."

# Check missingness and whether GSC/GA4 data was actually available.
missingness = con.sql(f"""
    SELECT
        COUNT(*) AS rows,

        ROUND(
            100 * AVG(CASE WHEN NOT gsc_data_available THEN 1 ELSE 0 END),
            2
        ) AS gsc_unavailable_pct,

        ROUND(
            100 * AVG(CASE WHEN NOT ga4_data_available THEN 1 ELSE 0 END),
            2
        ) AS ga4_unavailable_pct,

        ROUND(
            100 * AVG(CASE WHEN gsc_clicks IS NULL THEN 1 ELSE 0 END),
            2
        ) AS gsc_clicks_null_pct,

        ROUND(
            100 * AVG(CASE WHEN ga4_sessions IS NULL THEN 1 ELSE 0 END),
            2
        ) AS ga4_sessions_null_pct,

        ROUND(
            100 * AVG(CASE WHEN gsc_avg_position IS NULL THEN 1 ELSE 0 END),
            2
        ) AS position_null_pct

    FROM read_parquet('{march_path}')
""").df()

print("Missingness and data availability:")
display(missingness)

# Summarize the unequal amount of history across clients.
coverage = con.sql(f"""
    WITH client_days AS (
        SELECT
            client_hash_id,
            COUNT(DISTINCT report_date) AS observed_days
        FROM read_parquet('{march_path}')
        GROUP BY client_hash_id
    )
    SELECT
        MIN(observed_days) AS minimum_days,
        MEDIAN(observed_days) AS median_days,
        MAX(observed_days) AS maximum_days
    FROM client_days
""").df()

print("Observed days per client:")
display(coverage)

# Verify that the future outcome window exists.
april_path = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/month=2026-04/*.parquet"
)

outcome_window = con.sql(f"""
    SELECT
        COUNT(*) AS rows,
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date
    FROM read_parquet('{april_path}')
""").df()

print("Future outcome window:")
display(outcome_window)

assert str(outcome_window.loc[0, "first_date"].date()) == "2026-04-01"
assert str(outcome_window.loc[0, "last_date"].date()) == "2026-04-30"

print("Checks passed.")



March counts and window:


,row_count,first_date,last_date,client_count,content_count
0,9841378,2026-03-01,2026-03-31,55,331437


Grain violations (expected: 0 rows):


,report_date,client_hash_id,content_hash_id,copies


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

...

,minimum_days,median_days,maximum_days
0,9,31.0,31


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Future outcome window:


,rows,first_date,last_date
0,10424730,2026-04-01,2026-04-30


Checks passed.


## 4. Data limits

Client history is unbalanced. Four of 55 clients have fewer than 31 observed March days, so the same date filter does not provide the same amount of history for every client.

Some rows contain GSC data without available GA4 data. This occurred in 1,718,348 rows, or 17.46% of the March partition. Therefore, zero or non-null analytics values cannot always be interpreted as measured activity; the availability flags must be checked.

The March feature window and April outcome window do not overlap. Future measurements must still remain excluded from features. If I later use the fixed 90-day query table, I must check its window separately because it may overlap the outcome period.

The decline label is a defined proxy. It can identify measured, directional changes in clicks, but it cannot prove that content quality caused the decline or that refreshing the content would improve performance. The results provide decision support for human review, not a causal conclusion or automatic action.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 4: verify important data limits

limits = con.sql(f"""
    WITH client_history AS (
        SELECT
            client_hash_id,
            COUNT(DISTINCT report_date) AS observed_days
        FROM read_parquet('{march_path}')
        GROUP BY client_hash_id
    )
    SELECT
        (SELECT COUNT(*) FROM client_history) AS clients,
        (SELECT COUNT(*) FROM client_history
         WHERE observed_days < 31) AS clients_with_incomplete_month,

        COUNT(*) AS rows,

        COUNT_IF(
            gsc_data_available AND NOT ga4_data_available
        ) AS gsc_available_but_ga4_unavailable,

        ROUND(
            100.0 * COUNT_IF(
                gsc_data_available AND NOT ga4_data_available
            ) / COUNT(*),
            2
        ) AS gsc_only_row_pct

    FROM read_parquet('{march_path}')
""").df()

display(limits)

# Confirm that feature and outcome windows do not overlap.
feature_end = window_check.loc[0, "last_date"].date()
outcome_start = outcome_window.loc[0, "first_date"].date()

print("Feature window ends:", feature_end)
print("Outcome window starts:", outcome_start)

assert feature_end < outcome_start, "Feature and outcome windows overlap."

print("No calendar overlap was found.")

,clients,clients_with_incomplete_month,rows,gsc_available_but_ga4_unavailable,gsc_only_row_pct
0,55,4,9841378,1718348.0,17.46


Feature window ends: 2026-03-31
Outcome window starts: 2026-04-01
No calendar overlap was found.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.